# Phase 5 Pair #4 — n=10 Graduation Retrain (Colab)

**Purpose.** Test the pair #4 mechanism (metastability ~ replay-prioritization) at n=10 seeds against the pre-committed falsification criteria in `notes/notes/2026-05-20-metastability-replay-prioritization-dynamic-form.md`.

**Two same-seed conditions per seed** (the κ=0 control is run on identical seeds, not a different sample):

| Condition | metastability_obs_rate (μ_obs) | metastability_gain (κ) | metastability_replay_decay (μ_rep) |
|---|---|---|---|
| `kappa0_control` | 0.05 | **0.0** | 0.5 |
| `pair4_active`   | 0.05 | **2.0** | 0.5 |

Why this layout: μ_obs=0.05 (m_i EMA halflife ≈ 14 retrievals) and μ_rep=0.5 (one replay drains m by half) are non-zero in BOTH conditions to keep the bookkeeping identical; the only difference is κ (the priority gain). At κ=0 the priority composition is bit-identical to the pre-pivot baseline (load-bearing for the falsification baseline — verified in `tests/test_phase5_metastability.py::TestKappaZeroPreservesPriority`).

**Pre-committed pass criteria.** All four must hold:
1. **Phase 5 headline (graduation):** Δ meta_stable_rate at W=3, computed as `mean(rate_pair4 − rate_kappa0)` across the 10 same-seed pairs, with bootstrap 95% CI disjoint from zero AND Δ ≤ −0.1.
2. **D1 non-regression (Phase 4 preservation):** Δms_w3 ≤ −0.5 with CI disjoint from zero in the pair4_active runs.
3. **Replay-store dynamics sanity:** metastability_ema std/mean > 0.1 within first 500 steps (m_i becomes non-uniform).
4. **Substrate non-regression:** d_eff at step 1800 ≥ 25 in pair4_active.

**Seeds.** `[17, 11, 23, 1, 2, 3, 5, 7, 13, 29]`.

**Wall time estimate.** 20 parallel workers on H100 NVL ≈ 8–12 min.

**Colab gotchas:**
- ALWAYS pass `--device cuda` (substrate auto-detect is MPS/CPU only).
- Parent kernel must NOT touch CUDA before launching workers.
- Pre-warm the HuggingFace wikitext cache before the parallel launch.

In [ ]:
# 1. Clone, verify pair #4 implementation is present.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

import subprocess
markers = [
    ('def update_metastability', 'src/energy_memory/phase4/consolidation.py', 'm_i EMA method'),
    ('metastability_obs_rate', 'src/energy_memory/phase4/consolidation.py', 'config knob mu_obs'),
    ('metastability_gain', 'src/energy_memory/phase4/replay_loop.py', 'priority gain kappa'),
    ('metastability_replay_decay', 'src/energy_memory/phase4/replay_loop.py', 'pay-down mu_rep'),
    ('--metastability-obs-rate', 'experiments/19_phase34_integrated.py', 'exp 19 CLI'),
    ('weights_tensor=final_weights.detach()', 'src/energy_memory/memory/torch_hopfield.py', 'weights_tensor on result'),
    ('weights_tensor=final_weights.detach()', 'src/energy_memory/phase4/trajectory.py', 'weights_tensor on traced'),
]
for marker, fpath, label in markers:
    r = subprocess.run(['grep', '-n', '-e', marker, fpath], capture_output=True, text=True)
    status = 'OK' if r.returncode == 0 else 'MISSING'
    print(f'{status:8s} {label}: {marker}')
    if r.stdout:
        print(f'  {r.stdout.strip().splitlines()[0]}')

In [ ]:
# 2. Mount Drive + stage the phase3c codebook.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets

In [ ]:
# 4. CPU-only sanity. Parent must NOT touch CUDA.
#
# Just verify the codebook loads. The pair #4 unit tests were validated
# pre-commit (274 tests pass locally); cell 1's grep markers already
# confirm the new code paths are present in this checkout.
import sys, os, gc
sys.path.insert(0, 'src')

from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)
del cb; gc.collect()

# Tiny inline smoke: instantiate ConsolidationState with the new pair #4
# config and verify the new attributes / methods exist. This catches an
# accidental rollback without invoking unittest.
from energy_memory.phase4.consolidation import ConsolidationConfig, ConsolidationState
import torch
cfg = ConsolidationConfig(m=4, metastability_obs_rate=0.05)
state = ConsolidationState(cfg, device='cpu')
for _ in range(3):
    state.add_pattern(novelty_strength=1.0)
state.update_metastability(torch.tensor([0.5, 0.3, 0.2]))
state.metastability_payback(0, factor=0.5)
assert state.metastability_ema.shape == (3,), 'metastability_ema shape wrong'
assert hasattr(cfg, 'metastability_obs_rate'), 'config missing metastability_obs_rate'
print('pair #4 inline smoke: PASS (m_i =', state.metastability_ema.tolist(), ')')

from energy_memory.phase4.replay_loop import ReplayConfig
rc = ReplayConfig(metastability_gain=2.0, metastability_replay_decay=0.5)
assert rc.metastability_gain == 2.0
assert rc.metastability_replay_decay == 0.5
print('ReplayConfig pair #4 fields: PASS')

In [ ]:
# 4b. Pre-warm the wikitext cache so subprocesses do not race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
del splits; gc.collect()
print('cache warmed.')

In [ ]:
# 4c. GPU info (still no CUDA init in parent).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 5. Launch 20 parallel workers - 10 seeds x 2 conditions.
#
# Each worker is its own subprocess. The kappa=0 control and kappa>0 active
# condition for the same seed get IDENTICAL CLI args except --metastability-gain.

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
N_CUES = 3000
RUN_TAG = 'phase5_pair4_n10'

# Pre-committed parameters (locked - DO NOT tune after launch).
OBS_RATE = 0.05
REP_DECAY = 0.5
KAPPA_ACTIVE = 2.0
# A+B substrate prerequisites (same as the pair #4 baseline substrate).
ALPHA_ANTI = 1.0
COVERAGE_LAMBDA = 1.0
COVERAGE_EMA_RATE = 0.01
REPULSION_STEP_SIZE = 100.0

import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path(f'reports/{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

CONDITIONS = [
    ('kappa0_control', 0.0),
    ('pair4_active',   KAPPA_ACTIVE),
]

def launch(seed, condition_tag, kappa):
    out_dir = f'reports/{RUN_TAG}_{condition_tag}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'{condition_tag}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [
        'python', 'experiments/19_phase34_integrated.py',
        '--device', 'cuda',
        '--updater-kind', 'hebbian',
        '--seed', str(seed),
        '--n-cues', str(N_CUES),
        '--store-threshold', '0.3',
        '--alpha-anti', str(ALPHA_ANTI),
        '--coverage-lambda', str(COVERAGE_LAMBDA),
        '--coverage-ema-rate', str(COVERAGE_EMA_RATE),
        '--repulsion-step-size', str(REPULSION_STEP_SIZE),
        '--metastability-obs-rate', str(OBS_RATE),
        '--metastability-gain', str(kappa),
        '--metastability-replay-decay', str(REP_DECAY),
        '--output-dir', out_dir,
    ]
    print(f'launching {condition_tag} seed={seed} (kappa={kappa}) -> {out_dir}')
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=os.environ.copy())
    return proc, logf, log_path

procs = []
for seed in SEEDS:
    for tag, kappa in CONDITIONS:
        procs.append(launch(seed, tag, kappa))

t0 = time.time()
remaining = list(range(len(procs)))
while remaining:
    still_alive = []
    for i in remaining:
        proc, logf, log_path = procs[i]
        rc = proc.poll()
        if rc is None:
            still_alive.append(i)
        else:
            logf.close()
            mins = (time.time() - t0) / 60
            print(f'worker {i} done (rc={rc}) at {mins:.1f} min   log={log_path}')
    remaining = still_alive
    if remaining:
        time.sleep(30)
print(f'ALL DONE in {(time.time()-t0)/60:.1f} min')

In [ ]:
# 5b. EMERGENCY kill. Interrupt cell 7 first, then run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  pid err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 6. Aggregate the headline: Delta meta_stable_rate at W=3 across same-seed pairs.
#
# JSON structure (from experiments/19_phase34_integrated.py):
#   run = {"config": {...}, "results": {
#     "baseline_static":  [eval_row, ...],
#     "phase3_reencode":  [eval_row, ...],
#     "phase3_phase4":    [eval_row, ...],
#   }}
# Each eval_row has top-level keys top1/topk/cap_t_05/meta_stable_w{2,3,4}/
# cues_seen/condition; death_diag is a nested dict keyed by scale-as-string.
import json
import numpy as np
from pathlib import Path

SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
RUN_TAG = 'phase5_pair4_n10'

def load_run(condition_tag, seed):
    p = Path(f'reports/{RUN_TAG}_{condition_tag}_seed{seed}/phase34_results.json')
    if not p.exists():
        return None
    return json.loads(p.read_text())

def w3_meta_stable_rate_at_last_eval(run):
    if run is None:
        return None
    rows = run.get('results', {}).get('phase3_phase4', [])
    if not rows:
        return None
    last = max(rows, key=lambda r: r.get('cues_seen', 0))
    return last.get('meta_stable_w3')

pairs = []
for s in SEEDS:
    ctl = load_run('kappa0_control', s)
    act = load_run('pair4_active', s)
    r_ctl = w3_meta_stable_rate_at_last_eval(ctl)
    r_act = w3_meta_stable_rate_at_last_eval(act)
    if r_ctl is None or r_act is None:
        print(f'seed {s}: MISSING (ctl={r_ctl}, act={r_act})')
        continue
    delta = r_act - r_ctl
    pairs.append((s, r_ctl, r_act, delta))
    print(f'seed {s}: kappa0={r_ctl:.4f}  pair4={r_act:.4f}  delta={delta:+.4f}')

deltas = np.array([p[3] for p in pairs])
print(f'\n=== Headline: Delta meta_stable_rate at W=3, n={len(deltas)} same-seed pairs ===')
print(f'  mean delta = {deltas.mean():+.4f}')
print(f'  std  delta = {deltas.std(ddof=1):.4f}')

rng = np.random.default_rng(2026)
boots = [rng.choice(deltas, size=len(deltas), replace=True).mean() for _ in range(10_000)]
ci_lo, ci_hi = np.quantile(boots, [0.025, 0.975])
print(f'  bootstrap 95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]')

PASS_THRESHOLD = -0.10
ci_disjoint_neg = ci_hi < 0.0
hits_threshold = deltas.mean() <= PASS_THRESHOLD
print()
print(f'PASS criteria:')
print(f'  bootstrap CI disjoint from zero (upper < 0): {"PASS" if ci_disjoint_neg else "FAIL"}')
print(f'  mean delta <= {PASS_THRESHOLD:.2f}:                       {"PASS" if hits_threshold else "FAIL"}')
print(f'  overall:                                       {"PASS" if (ci_disjoint_neg and hits_threshold) else "FAIL"}')

In [ ]:
# 7. Drill-downs: m_i distribution sanity + n_patterns + D1 non-regression.
import numpy as np

def first_eval_row(run):
    rows = (run or {}).get('results', {}).get('phase3_phase4', [])
    if not rows:
        return None
    return min(rows, key=lambda r: r.get('cues_seen', 0))

def last_eval_row(run):
    rows = (run or {}).get('results', {}).get('phase3_phase4', [])
    if not rows:
        return None
    return max(rows, key=lambda r: r.get('cues_seen', 0))

def death_diag_for_scale(row, scale):
    dd = (row or {}).get('death_diag', {}) or {}
    return dd.get(str(scale)) or dd.get(scale) or {}

print('=== Drill-down: m_i distribution (audit-binding sanity check) ===')
print('  At first eval (~step 500), std/mean of metastability_ema should')
print('  be > 0.1 for pair4_active; otherwise the mechanism is not')
print('  differentiating between atoms.\n')
for s in SEEDS:
    act = load_run('pair4_active', s)
    if not act:
        continue
    row = first_eval_row(act)
    if not row:
        continue
    dd = death_diag_for_scale(row, 3)
    m_mean = dd.get('metastability_ema_mean', 0.0) or 0.0
    m_std = dd.get('metastability_ema_std', 0.0) or 0.0
    cv = (m_std / m_mean) if m_mean > 1e-6 else 0.0
    print(f'  seed {s}: m_mean={m_mean:.4f}  m_std={m_std:.4f}  CV={cv:.2f}  '
          f'({"PASS" if cv > 0.1 else "FAIL uniform"})')

print('\n=== Drill-down: substrate survival (n_patterns at W=4 final eval) ===')
print('  D_eff is not currently surfaced per-eval; n_patterns is the')
print('  available substrate-size proxy. Phase 4 D1 graduation regime')
print('  had W=4 n_patterns ~5-30 post mass-death.\n')
for s in SEEDS:
    act = load_run('pair4_active', s)
    if not act:
        continue
    row = last_eval_row(act)
    if not row:
        continue
    dd = death_diag_for_scale(row, 4)
    n_patterns = dd.get('n_patterns', None)
    print(f'  seed {s}: W=4 final n_patterns = {n_patterns}')

print('\n=== Drill-down: D1 non-regression (Phase 4 meta_stable_w3 still suppressed) ===')
print('  pair4_active should not regress meta_stable_w3 substantially')
print('  vs the Phase 4 D1 graduation baseline.\n')
for s in SEEDS:
    ctl = load_run('kappa0_control', s)
    act = load_run('pair4_active', s)
    if not (ctl and act):
        continue
    ms_ctl = (last_eval_row(ctl) or {}).get('meta_stable_w3', None)
    ms_act = (last_eval_row(act) or {}).get('meta_stable_w3', None)
    if ms_ctl is None or ms_act is None:
        continue
    print(f'  seed {s}: kappa0 ms_w3={ms_ctl:.4f}  pair4 ms_w3={ms_act:.4f}')

In [ ]:
# 8. Copy results back to Drive.
import shutil, os
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 29]
RUN_TAG = 'phase5_pair4_n10'
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for s in SEEDS:
    for tag in ('kappa0_control', 'pair4_active'):
        src = f'reports/{RUN_TAG}_{tag}_seed{s}'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst}/{RUN_TAG}_{tag}_seed{s}', dirs_exist_ok=True)
shutil.copytree(f'reports/{RUN_TAG}_colab', f'{dst}/{RUN_TAG}_colab', dirs_exist_ok=True)
print('results copied to', dst)
!ls {dst} | grep phase5_pair4 | head -25